# Advanced Iterators: itertools, Windows, Sort-Merge Joins, Lazy Pipelines

## What you'll practice

Level 3 introduced generators; here you build streaming algorithms with them: running aggregates, sliding windows, run-length grouping, a sort-merge join over two
sorted streams, and a lazy, chainable pipeline API that behaves like Spark RDDs. (Decorators and context managers were covered hands-on in Level 5 — revisit
`05_functions_and_oop/03_decorators` and `05_context_managers` if those felt shaky.)

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/advanced/test_10_01_generators_itertools.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/10_advanced_python/01_generators_itertools.ipynb`. Don't open it until you've tried.

## Exercise 1 — 🟢 Easy · Running maximum

**Implement:** `running_max(values) -> Iterator`

Yield the running maximum of any iterable, lazily, using `itertools.accumulate`.

**Input:** `values`

**Expected output:** `Iterator`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
list(running_max([3, 1, 4, 1, 5]))   # [3, 3, 4, 4, 5]
```

**Check:** `pytest tests/advanced/test_10_01_generators_itertools.py -k test_running_max`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · Sliding windows

**Implement:** `windowed(iterable, n: int) -> Iterator[tuple]`

Yield overlapping tuples of length `n` (`windowed([1,2,3,4], 3)` → `(1,2,3), (2,3,4)`). Inputs shorter than `n` yield nothing. Use a `deque(maxlen=n)`; work on any iterable lazily. `n < 1` → `ValueError`.

**Input:** `iterable, n: int`

**Expected output:** `Iterator[tuple]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_windowed` in `tests/advanced/test_10_01_generators_itertools.py`.

**Check:** `pytest tests/advanced/test_10_01_generators_itertools.py -k test_windowed`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Run-length grouping with groupby

**Implement:** `status_runs(events: list[dict]) -> list[tuple[str, int, str, str]]`

Events (sorted by `ts`) carry a `status`. Collapse consecutive identical statuses into runs: `(status, count, first_ts, last_ts)`. A status that reappears later starts a new run.

**Input:** `events: list[dict]`

**Expected output:** `list[tuple[str, int, str, str]]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example**

```python
status_runs([{"ts": "t1", "status": "UP"}, {"ts": "t2", "status": "UP"}, {"ts": "t3", "status": "DOWN"}, {"ts": "t4", "status": "UP"}])
# [('UP', 2, 't1', 't2'), ('DOWN', 1, 't3', 't3'), ('UP', 1, 't4', 't4')]
```

**Check:** `pytest tests/advanced/test_10_01_generators_itertools.py -k test_status_runs`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · Streaming sort-merge join

**Implement:** `merge_join(left, right, key) -> Iterator[tuple[dict, dict]]`

Both inputs are iterables of dicts **sorted by** `key` (a function). Yield `(l, r)` for every matching pair (inner join); duplicate keys on both sides produce the cartesian product
for that key. Consume each input once, holding only one key's group from each side in memory. Raise `ValueError` if either input turns out not to be sorted.

**Input:** `left, right, key`

**Expected output:** `Iterator[tuple[dict, dict]]`

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_merge_join, test_merge_join_unsorted, test_merge_join_large_streaming` in `tests/advanced/test_10_01_generators_itertools.py`.

**Check:** `pytest tests/advanced/test_10_01_generators_itertools.py -k "test_merge_join or test_merge_join_unsorted or test_merge_join_large_streaming"`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · A lazy, chainable pipeline API (mini RDD) · 🧩 Mini-project

**Implement:** `class Stream`

`Stream(source)` where `source` is an iterable **or a zero-arg callable returning an iterable** (so the stream can be re-run). Transformations return a **new** `Stream` and run
nothing: `map(f)`, `filter(p)`, `flat_map(f)`, `distinct(key=None)`, `batch(n)` (yields lists), `take(n)` (stop early). Actions run the pipeline: `collect()` → list, `count()`,
`reduce(f, initial)`, `first(default=None)`, `group_by_key(key, value=lambda x: x)` → dict of lists. Calling an action twice on a callable-source stream recomputes from scratch;
on a one-shot iterator source, the second action sees an exhausted stream (that's documented behaviour, like Spark's need to `cache()`). Keep everything lazy: `take(3)` on an
infinite source must terminate.

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library only, unless the statement says otherwise.

**Example:** the tests are executable examples — read `test_stream_lazy_and_chainable, test_stream_infinite_and_branching, test_stream_other_ops` in `tests/advanced/test_10_01_generators_itertools.py`.

**Check:** `pytest tests/advanced/test_10_01_generators_itertools.py -k "test_stream_lazy_and_chainable or test_stream_infinite_and_branching or test_stream_other_ops or test_stream_one_shot_source"`

**YOUR CODE ↓**